# S5.G2R-1 · A_secure 超限补训到 150 棵：补训记录与打分

> S5.G2 按 DR-M5-010 · D-3 在 6 小时上限处停在 141 棵，A_secure 记为「超出单次拟合上限、未完成」——**这一预注册结论不改**。所有者 2026-11-02 指示补训剩余部分（DR-M5-011）：141 棵检查点复制到新目录（S5.G2 的目录不动），用同一组件 `g3_ssxgb.launch` 续训到 150 棵，不经上限检查；参数、端口、SPU 协议与域全部沿用 S5.G2。

In [1]:
import hashlib
import json
import os
import pathlib
import subprocess
import sys

import numpy as np
import pandas as pd
import yaml

HASH_PREFIX_LEN = 12
BYTES_PER_GIB = 1024 ** 3
D4_MIN_ABS_DIFF = 0.005          # DR-M5-010 · D-4：「可分辨」但差值小于此值时降级
D1_MAX_SLOWDOWN = 1.5            # DR-M5-011 · D-1：补训段每棵耗时超过 S5.G2 均值的此倍数则补训作废

REPO_ROOT = pathlib.Path.cwd()
while not (REPO_ROOT / "AGENTS.md").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

CONFIG_PATH = REPO_ROOT / "modules/m5_modeling/configs/s5_g2r_ssxgb_full150.yaml"
rr = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))
BASE_PATH = REPO_ROOT / rr["base_config"]
run = yaml.safe_load(BASE_PATH.read_text(encoding="utf-8"))
PRE_PATH = REPO_ROOT / run["preregistration"]
pre = yaml.safe_load(PRE_PATH.read_text(encoding="utf-8"))
fz = pre["freeze"]
ad = fz["arm_details"]
SSXGB_PATH = REPO_ROOT / "modules/m5_modeling/components/g3_ssxgb.py"


def sha(path, n=HASH_PREFIX_LEN):
    return hashlib.sha256(pathlib.Path(path).read_bytes()).hexdigest()[:n]


git_sha = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True, cwd=REPO_ROOT).strip()
dirty = subprocess.run(["git", "status", "--porcelain"], capture_output=True, text=True,
                       cwd=REPO_ROOT).stdout.strip() != ""
print("config:", CONFIG_PATH.relative_to(REPO_ROOT), "sha256:" + sha(CONFIG_PATH))
print("base config:", BASE_PATH.relative_to(REPO_ROOT), "sha256:" + sha(BASE_PATH))
print("preregistration:", PRE_PATH.relative_to(REPO_ROOT), "sha256:" + sha(PRE_PATH), "| status:", pre["status"])
print("决策记录:", rr["decision_record"], "| 偏离:", rr["override_note"])
print("seed:", rr["seed"], "| seeds:", rr["seeds"])
print("git:", git_sha + ("（工作区含本步未提交改动）" if dirty else ""))
print("step:", rr["step_id"], "|", rr["purpose"])
print("SS-XGB 组件:", SSXGB_PATH.relative_to(REPO_ROOT), "sha256:" + sha(SSXGB_PATH))
mismatch = [p for p, h in fz["code_sha256"].items() if hashlib.sha256((REPO_ROOT / p).read_bytes()).hexdigest() != h]
print("冻结代码指纹核对：", "一致" if not mismatch else "不一致 " + str(mismatch))
assert pre["status"] == "frozen" and not mismatch
PRIV = pathlib.Path(os.environ[run["private_root_env"]])
SEED0 = rr["seeds"][0]
MB, NJ = ad["xgboost_max_bin"], ad["n_jobs"]

config: modules/m5_modeling/configs/s5_g2r_ssxgb_full150.yaml sha256:16ec2b9257f2
base config: modules/m5_modeling/configs/s5_g2_ssxgb_run.yaml sha256:0ad22448d9fc
preregistration: modules/m6_evaluation/configs/s6_p3_g3_preregistration.yaml sha256:c0a4bbfc9709 | status: frozen
决策记录: registry/decision_records/DR-M5-011.yaml | 偏离: owner_override_cap（DR-M5-011）
seed: 11 | seeds: [11, 22, 33, 44, 55, 66]
git: 75016e5（工作区含本步未提交改动）
step: S5.G2R | 所有者指示把 A_secure 从 141 棵续训到 150 棵（超出预注册单次拟合上限），按 S6.P3 口径报告 A_secure − A 并标注偏离
SS-XGB 组件: modules/m5_modeling/components/g3_ssxgb.py sha256:0f55ee4a8826
冻结代码指纹核对： 一致


In [2]:
from modules.m5_modeling.components import g3_sgb as SG
from modules.m5_modeling.components import g3_ssxgb as SX

TUNING = json.loads((PRIV / run["tuning_file"]).read_text(encoding="utf-8"))
params = SG.sgb_params(TUNING["A"]["best"], ad["sgb_sketch_eps"], run["seed"])
TOTAL, CAP = params["num_boost_round"], pre["compute_caps"]["max_wall_clock_hours_per_fit"]
SRC, CKPT = PRIV / rr["source_ckpt_dir"], PRIV / rr["ckpt_dir"]
before = [json.loads(x) for x in (CKPT / "chunks_s5g2.jsonl").read_text(encoding="utf-8").splitlines()]
extra = [json.loads(x) for x in (CKPT / "chunks.jsonl").read_text(encoding="utf-8").splitlines()]
assert (CKPT / "chunks_s5g2.jsonl").read_bytes() == (SRC / "chunks.jsonl").read_bytes()   # S5.G2 的分段记录原样
assert all(c["component_sha256"] == sha(SSXGB_PATH, None) and c["params"] == params for c in before + extra)
res = [c for c in extra if c["event"] == "result"]
assert len(res) == sum(c["event"] == "launch" for c in extra) == 1
cols = ["started", "resume_from", "stop_at", "trees_done", "wall_seconds", "exit_codes", "note"]
print("补训记录（g3_ssxgb.launch，不经上限检查；组件指纹与参数与 S5.G2 各段一致）：")
print(pd.DataFrame([{k: c[k] for k in cols} for c in res]).round(1).to_string(index=False))
old = SX.load_progress(str(SRC), SX.LABEL_PARTY)
prog = {p: SX.load_progress(str(CKPT), p) for p in SX.PARTIES}
T0 = old["trees_done"]
assert prog["neidi"]["trees_done"] == prog["hk"]["trees_done"] == TOTAL
assert prog[SX.LABEL_PARTY]["tree_seconds"][:T0] == old["tree_seconds"]          # 前 T0 棵的记录未改
COST = SX.cost_summary(prog[SX.LABEL_PARTY], TOTAL, CAP)
tree_s = np.asarray(prog[SX.LABEL_PARTY]["tree_seconds"])
slow = float(tree_s[T0:].mean() / tree_s[:T0].mean())
peak = max(SX.cost_summary(prog[p], TOTAL, CAP)["peak_rss_mb"] for p in SX.PARTIES)
print("补训 %d → %d 棵：每棵平均 %.1f 秒（S5.G2 前 %d 棵 %.1f 秒，比值 %.2f，作废线 %.1f）；补训段回放 %.0f 秒"
      % (T0, TOTAL, tree_s[T0:].mean(), T0, tree_s[:T0].mean(), slow, D1_MAX_SLOWDOWN,
         prog[SX.LABEL_PARTY]["chunks"][-1]["replay_seconds"]))
print("150 棵拟合墙钟 %.0f 秒（%.2f 小时，预注册上限 %s 小时）；在上限内：%s"
      % (COST["fit_seconds"], COST["fit_seconds"] / SX.SECONDS_PER_HOUR, CAP, COST["within_cap"]))
print("续训开销合计 %.0f 秒；回环发送每棵约 %.1f GiB；两方进程峰值内存 %.0f MB"
      % (COST["resume_overhead_seconds"], COST["lo_tx_bytes"] / TOTAL / BYTES_PER_GIB, peak))
assert slow <= D1_MAX_SLOWDOWN

补训记录（g3_ssxgb.launch，不经上限检查；组件指纹与参数与 S5.G2 各段一致）：
                 started  resume_from  stop_at  trees_done  wall_seconds            exit_codes                          note
2026-11-02T15:32:06+0800          141      150         150        2166.8 {'hk': 0, 'neidi': 0} owner_override_cap（DR-M5-011）
补训 141 → 150 棵：每棵平均 186.8 秒（S5.G2 前 141 棵 152.3 秒，比值 1.23，作废线 1.5）；补训段回放 431 秒
150 棵拟合墙钟 23172 秒（6.44 小时，预注册上限 6 小时）；在上限内：False
续训开销合计 1321 秒；回环发送每棵约 32.1 GiB；两方进程峰值内存 3346 MB


**读法**：补训 9 棵，每棵平均 186.8 秒，是 S5.G2 前 141 棵均值 152.3 秒的 1.23 倍，低于 DR-M5-011 · D-1 的作废线 1.5 倍（同一台机器当日更慢，回放 141 棵也用了 431 秒）。150 棵的拟合墙钟累计 23,172 秒（6.44 小时），超过预注册上限 6 小时；续训开销（各段重做准备与回放）合计 1,321 秒，单列。两方每棵树经本机回环合计发送约 32 GiB，进程峰值内存约 3.3 GB，与 S5.G2 相同。

In [3]:
SCORES = PRIV / rr["ssxgb_scores_file"]
full = {"python": sys.executable, "repo": str(REPO_ROOT), "arrays": str(PRIV / run["arrays_file"]),
        "prefix": "dev_eval", "params": params, "fed_ports": run["fed_ports"], "spu_ports": run["spu_ports"],
        "protocol": run["spu_protocol"], "field": run["spu_field"], "ckpt": str(CKPT)}
codes = SX.launch("predict", full, TOTAL, TOTAL, out=str(SCORES))
assert all(v == 0 for v in codes.values())
pred = np.load(SCORES)
assert pred.shape == np.load(PRIV / run["arrays_file"])["dev_eval__y"].shape
last = SX.load_progress(str(CKPT), SX.LABEL_PARTY)["predict"][-1]
print("A_secure（%d 棵）对开发评估集 %d 位客户打分，分数只揭示给内地；耗时 %.0f 秒" % (last["trees"], len(pred), last["seconds"]))

A_secure（150 棵）对开发评估集 20351 位客户打分，分数只揭示给内地；耗时 89 秒


**读法**：150 棵模型对开发评估集 20,351 位客户打分用时 89 秒，分数只揭示给内地；逐行分数只存私有目录，不入仓库。